# LLM Coding Notebook — Audience Analysis (Nigeria)

**Norman Lear Center × Gates Foundation — Manfluencer Project**

Codes a stratified random sample of audience comments using `gpt-4o-mini` against the final codebook (`Codebooks/LLM Codebook/Nigeria Manfluencer Project - Final Coding Codebook.docx`).

## Inputs
- `Nigeria/Audience Analysis/Translated/audience_final_translated.parquet` — 417 manager-curated comments × 4 creators.

## Sampling
Stratified 50 per creator → **200 comments**, balanced 50/50 progressive (Banky + Deyemi) vs regressive (Agba + Shola). Seed fixed at 42 for reproducibility.

## Coding produced per row
Front summary columns (LLM-generated, concise):
- **Themes** — up to 3 from the 13-theme final list + Unclear
- **Sentiment** — Positive / Negative / Neutral / Unclear (matches human codebook Q1)
- **Emotion Detection** — discrete emotion (Q2 vocabulary)
- **Tone** — rhetorical register (NEW dimension; values defined in codebook §7)

Plus Q1–Q21h matching the human audience codebook exactly.

## Output
- `Codebooks/LLM Codebook/LLM Coding - Audience Analysis.xlsx`

## Cost
~200 rows × gpt-4o-mini ≈ **\$0.15**, ~5 minutes wall time.

## 0 — Setup

In [1]:
from __future__ import annotations
import asyncio, hashlib, json, os, re, random
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm as atqdm

ROOT = Path.cwd().resolve()
while ROOT.name != 'Gates-Manfluencer-Project' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert ROOT.name == 'Gates-Manfluencer-Project'

load_dotenv(ROOT / '.env')
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY missing'

AUD_PARQUET = ROOT / 'Nigeria' / 'Audience Analysis' / 'Translated' / 'audience_final_translated.parquet'
OUT_DIR     = ROOT / 'Codebooks' / 'LLM Codebook'
OUT_XLSX    = OUT_DIR / 'LLM Coding - Audience Analysis.xlsx'
CACHE       = ROOT / 'temp' / 'llm_audience_coding_cache.parquet'
CACHE.parent.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL = 'gpt-4o-mini'
SEED  = 42
N_PER_CREATOR = 50      # 50 × 4 = 200 total
CONCURRENCY = 8

print(f'ROOT = {ROOT}')
print(f'model = {MODEL}, seed = {SEED}, n per creator = {N_PER_CREATOR} → total {N_PER_CREATOR*4}')

ROOT = /Users/sushildalavi/Desktop/NLC/Gates-Manfluencer-Project
model = gpt-4o-mini, seed = 42, n per creator = 50 → total 200


## 1 — Load + stratified sample

In [2]:
df = pd.read_parquet(AUD_PARQUET)
ORIENT = {'Banky Wellington':'progressive', 'Deyemi Okanlawon':'progressive',
          'Agba John Doe':'regressive', 'Shola':'regressive'}
df['orientation'] = df['creator'].map(ORIENT)
print(f'full audience: {len(df)} rows')
print(df['creator'].value_counts().to_string())

samples = []
for cr, sub in df.groupby('creator'):
    samples.append(sub.sample(n=N_PER_CREATOR, random_state=SEED))
sample = pd.concat(samples).reset_index(drop=True)
print(f'\nstratified sample: {len(sample)} rows')
print(sample.groupby(['orientation','creator']).size().to_string())

full audience: 417 rows
creator
Agba John Doe       161
Banky Wellington    110
Deyemi Okanlawon     80
Shola                66

stratified sample: 200 rows
orientation  creator         
progressive  Banky Wellington    50
             Deyemi Okanlawon    50
regressive   Agba John Doe       50
             Shola               50


## 2 — Split source_url into commenter + influencer URLs

X-platform rows have format `<commenter_url> (reply on <creator>'s post <og_url>)`. YouTube rows have a single video URL (used for both).

In [3]:
def split_urls(s: str) -> tuple[str, str]:
    if not isinstance(s, str): return ('', '')
    m = re.match(r'^(\S+)\s*\(reply on .*?(https?://\S+)\)\s*$', s)
    if m:
        return (m.group(1).strip(), m.group(2).rstrip(')').strip())
    # youtube / single-url case → use same url for both
    return (s.strip(), s.strip())

url_split = sample['source_url'].apply(split_urls)
sample['Commenter Post URL'] = url_split.apply(lambda x: x[0])
sample["Influencer's OG Post URL"] = url_split.apply(lambda x: x[1])
print(sample[['comment_id', 'Commenter Post URL', "Influencer's OG Post URL"]].head(8).to_string())

  comment_id                                        Commenter Post URL                            Influencer's OG Post URL
0    AGB_109        https://x.com/Ucee_Dave/status/1556958497509703680  https://x.com/jon_d_doe/status/1556739908810817536
1    AGB_112     https://x.com/Ladychelgift/status/1556940731071201280  https://x.com/jon_d_doe/status/1556739908810817536
2    AGB_150  https://x.com/QueenMotherFeyi/status/1557065455839354882  https://x.com/jon_d_doe/status/1556739908810817536
3    AGB_056         https://x.com/Mumbari2/status/1556880639164461056  https://x.com/jon_d_doe/status/1556739908810817536
4    AGB_097  https://x.com/IbekweEmmanue20/status/1556754010715181057  https://x.com/jon_d_doe/status/1556739908810817536
5    AGB_030        https://x.com/mz_imaima/status/1556922003306725382  https://x.com/jon_d_doe/status/1556739908810817536
6    AGB_105      https://x.com/Jo_4444real/status/1556975925631262720  https://x.com/jon_d_doe/status/1556739908810817536
7    AGB_052    

## 3 — Coding prompt (single LLM call per row → full structured JSON)

In [4]:
THEMES = [
    'Authority and Submission', 'Male Victimhood', 'Gender Grievance',
    'Sexual Morality', 'Relationship Tactics', 'Provider and Status',
    'Male Accountability', 'Egalitarian Partnership',
    'Gender-Based Violence and Consent', 'Trauma and Mental Health',
    'Self-Discipline', 'Marriage and Family', 'Faith and Moral Repair',
    'Unclear',
]

SENTIMENTS = ['Positive', 'Negative', 'Neutral', 'Unclear']

EMOTIONS = ['Joy', 'Happiness', 'Surprise', 'Anger', 'Fear', 'Contempt',
            'Sadness', 'Hope', 'Empathy', 'None of these']

TONES = ['Earnest', 'Sarcastic', 'Hostile', 'Humorous',
         'Empathetic', 'Authoritative', 'Detached']

NORMATIVE_ORIENTATIONS = ['Progressive', 'Regressive', 'Mixed', 'Unclear']

TARGETS = ['Men/boys', 'Women/girls', 'Feminists/modern women',
           'Children/family', 'Institutions/law/society', 'Creator/content',
           'Self/personal life', 'Mixed', 'Unclear']

PROMPT = f"""You are a senior research-grade content annotator coding Nigerian masculinity-related social-media AUDIENCE COMMENTS for the Norman Lear Center / Gates Foundation Manfluencer Project.

You are coding ONE comment at a time.

Read the full comment carefully before assigning any code.

Treat each field as a defensible research judgment, not a guess. Apply the codebook strictly. When in doubt, choose the more conservative option: Unclear, No, None of these, Neutral, or an empty string for conditional fields.

Do not invent labels. Use ONLY the exact options listed below.

Open-text fields must be specific and quote-grounded. Never write generic placeholders like "explains support", "the commenter agrees", or "the comment is about gender".

If a field is conditional, leave it as an empty string when the gating answer is No or not applicable.

────────────────────────────────────────────────────────────────────
PRIMARY AND SECONDARY THEMES
────────────────────────────────────────────────────────────────────
Pick exactly ONE primary_theme.

Pick up to TWO secondary themes:
- secondary_theme_1 may be empty if there is no strong secondary theme.
- secondary_theme_2 may be empty if there is no second strong secondary theme.
- Do not duplicate the primary theme.
- Do not duplicate secondary themes.
- If primary_theme = "Unclear", both secondary fields must be empty.
- Use "Unclear" only if no real theme fits.

Allowed theme labels (use these exact strings):

1. Authority and Submission — explicit hierarchy: headship, submission, control, surname, command, "men lead / women serve". Requires explicit hierarchy/control language. Do NOT use for general family talk.
2. Male Victimhood — men framed as exploited, disadvantaged, losing, harmed by women, courts, child support, false accusations, divorce, or equality. Focus on male loss.
3. Gender Grievance — generalized distrust of women / feminists / equality framed as scam or threat; gender-war framing. Requires generalization across women as a class, not specific criticism of one person.
4. Sexual Morality — cheating, body count, abortion, BBL/body policing, sexual respectability, female desirability, sexual double standards. If violence/consent is central, use Gender-Based Violence and Consent instead.
5. Relationship Tactics — tactical dating advice (scarcity, pursuit, availability, options, rejection, masculine-frame instructions, how to keep/control/desire a partner). NOT generic relationship praise or values talk.
6. Provider and Status — money/income/career/status/respectability as proof of manhood or relationship worth. Use when economic/status value is central.
7. Male Accountability — "men must change", "hold men accountable", men's behavior as the problem to solve, refusal of deflection. Distinct from Gender-Based Violence and Consent (violence-specific) and Egalitarian Partnership (reciprocity).
8. Egalitarian Partnership — mutual respect, shared money/parenting, listening, allyship, healthy reciprocity, men supporting women without hierarchy. Requires a relational norm, not just generic praise.
9. Gender-Based Violence and Consent — rape, consent, abuse, victim stigma, false accusations, prosecution, justice, institutional response to abuse. Both anti-rape advocacy and false-accusation pushback code here when rooted in the violence/consent frame.
10. Trauma and Mental Health — trauma, depression, grief, healing, vulnerability, psychological harm, therapy, emotional openness. Inner life must be substantive, not figurative ("this is painful" rhetorical does not qualify).
11. Self-Discipline — personal responsibility, restraint, growth, learning. Constructive self-improvement only. NOT discipline-as-controlling-women.
12. Marriage and Family — marriage / divorce / family / fatherhood / motherhood / household duty as the OBJECT of discussion (not just the setting where another norm appears). Apply strictly.
13. Faith and Moral Repair — explicit faith / scripture / God / prayer / sin / testimony tied to masculinity, marriage, healing, or moral conduct. Generic morality without religious frame does NOT qualify.
14. Unclear — uncodable / off-topic / low-signal.

Disambiguation (priority order, first applicable wins):
- Gender-Based Violence and Consent beats Sexual Morality when violence/consent is central.
- Authority and Submission beats Marriage and Family when hierarchy/control is central.
- Specificity beats Marriage and Family — use the more specific theme when marriage is just the setting.
- Faith and Moral Repair requires explicit religious vocabulary.
- Relationship Tactics requires tactical content; values talk goes elsewhere.
- Male Victimhood = men-as-harmed; Gender Grievance = women-as-threat. Both can co-occur (one primary, one secondary).

────────────────────────────────────────────────────────────────────
MASCULINITY IDENTITY (scope marker)
────────────────────────────────────────────────────────────────────
masculinity_identity: "Yes" / "No"

"Yes" if the comment directly discusses men, boys, manhood, masculinity, male socialization, male identity, male behavior, or male roles. Stricter than q4.

────────────────────────────────────────────────────────────────────
NORMATIVE ORIENTATION
────────────────────────────────────────────────────────────────────
normative_orientation: {' / '.join(NORMATIVE_ORIENTATIONS)}

Progressive — challenges hierarchy, supports equality, supports male accountability, empathy, consent, healthy vulnerability, or shared partnership.
Regressive — reinforces hierarchy, misogyny, rigid gender roles, male dominance, female submission, anti-feminist grievance, sexual double standards, or victim-blaming.
Mixed — both progressive and regressive elements present.
Unclear — no clear ideological direction.

Do NOT infer orientation from the creator. Code only the comment itself.

────────────────────────────────────────────────────────────────────
TARGET OF CLAIM
────────────────────────────────────────────────────────────────────
target_of_claim: {' / '.join(TARGETS)}

Pick the main target being discussed, blamed, praised, advised, defended, or evaluated.

────────────────────────────────────────────────────────────────────
SENTIMENT
────────────────────────────────────────────────────────────────────
sentiment: {' / '.join(SENTIMENTS)}

Positive — favorable, approving, supportive, grateful, admiring.
Negative — critical, angry, sad, hostile, contemptuous, fearful, disapproving.
Neutral — descriptive, informational, balanced, no clear valence.
Unclear — too vague to determine.

q1 MUST exactly match sentiment.

────────────────────────────────────────────────────────────────────
EMOTION
────────────────────────────────────────────────────────────────────
emotion: {' / '.join(EMOTIONS)}

Pick the single dominant emotion expressed. Use "None of these" only when the comment is purely informational with no detectable emotional charge.

q2 MUST exactly match emotion.

────────────────────────────────────────────────────────────────────
TONE
────────────────────────────────────────────────────────────────────
tone: {' / '.join(TONES)}

Tone is HOW the message is delivered, distinct from emotion (what the speaker feels). A sarcastic comment can express Anger.

Earnest — sincere, direct, no irony.
Sarcastic — ironic, mocking, opposite-meaning surface.
Hostile — aggressive, attacking, confrontational, insulting.
Humorous — playful, joking, light-hearted (genuine humor, not sarcasm).
Empathetic — supportive, compassionate, validating.
Authoritative — didactic, prescriptive, lecturing.
Detached — neutral, observational, distant.

────────────────────────────────────────────────────────────────────
HUMAN CODEBOOK QUESTIONS (Q1–Q21h)
────────────────────────────────────────────────────────────────────
Match the human audience codebook EXACTLY. Use only the listed options.

q1 — overall sentiment ({' / '.join(SENTIMENTS)}). MUST equal sentiment.
q2 — primary emotional tone ({' / '.join(EMOTIONS)}). MUST equal emotion.
q3 — emotional response (multi-select; use only textually evidenced):
   "Feeling seen/understood", "Feeling unseen/misunderstood",
   "Feeling attacked", "Feeling objectified", "None of these"
q4 — mentions men/women/gender norms ("Yes" / "No")
q5 — sentiment toward men/masculinity (Positive / Negative / Neutral / Unclear / Does not mention men/masculinity)
q6 — sentiment toward women/femininity (Positive / Negative / Neutral / Unclear / Does not mention women/femininity)
q7 — main topic (multi-select):
   "The speaker/creator of the content/influencer/the content",
   "Politics/social issues", "Dating/relationships/marriage", "Money/status",
   "Fitness", "Media/video games", "Mental health/emotions",
   "Gender roles/norms", "Other"
q7a — if "Other" in q7, one short phrase. Else empty string.
q8 — commenter stance (Supporting / Challenging / Neutral / Unclear)
q9 — if q8 = "Supporting", ONE sentence summarizing the specific reason given (quote or paraphrase). Else empty string.
q10 — if q8 = "Supporting", what need is being served (multi-select):
   "Entertainment/escapism", "Information seeking",
   "Connection/social interaction",
   "Self expression/identity construction", "Status seeking",
   "Documentation of events", "None of these apply"
   If q8 != "Supporting", use ["None of these apply"].
q11 — if q8 = "Challenging", ONE sentence summarizing the specific objection. Else empty string.
q12 — references personal experience ("Yes" / "No"). Yes only with first-person ("I", "my", "we") + specific event.
q13 — sexist or derogatory-to-women language ("Yes" / "No"). High bar — slurs, dehumanization, sweeping negative claims about women as a class. Mere disagreement is not enough.
q14 — indicates new knowledge acquired ("Yes" / "No"). Yes only if explicitly stated.
q14a — if q14 = "Yes", ONE sentence describing what was learned. Else empty.
q15 — indicates attitude changed ("Yes" / "No"). Yes only if explicitly described.
q15a — if q15 = "Yes", ONE sentence describing how. Else empty.
q16 — opinion reinforced ("No" / "Yes"). Yes if commenter says content confirmed prior belief.
q16a — if q16 = "Yes", ONE sentence stating the reinforced opinion. Else empty.
q17 — calls to action ("Yes" / "No"). Yes only if comment urges an action.
q17a — if q17 = "Yes", ONE phrase stating the action. Else empty.
q18 — shares info/fact/link/personal info ("Yes" / "No").
q18a — if q18 = "Yes", ONE phrase summarizing what was shared. Else empty.
q19 — advocates for cause/norm/standard ("No" / "Yes").
q19a — if q19 = "Yes", ONE phrase stating what is advocated. Else empty.
q20 — corrects something incorrect ("No" / "Yes").
q20a — if q20 = "Yes", ONE phrase summarizing what is corrected. Else empty.
q21 — self-identifies ("No" / "Yes"). Yes only if commenter explicitly states something about themselves.
q21a — profession mentioned ("No" / "Yes")
q21b — if q21a = "Yes", profession text. Else empty.
q21c — location mentioned ("No" / "Yes")
q21d — if q21c = "Yes", location text. Else empty.
q21e — race/ethnicity mentioned ("No" / "Yes")
q21f — if q21e = "Yes", race/ethnicity text. Else empty.
q21g — gender mentioned ("No" / "Yes")
q21h — if q21g = "Yes", one of: "Female" / "Male" / "Non-binary" / "Other" / "Unclear". Else empty.

────────────────────────────────────────────────────────────────────
QUALITY CHECKLIST (silent, before returning)
────────────────────────────────────────────────────────────────────
1. Read the entire comment.
2. primary_theme defensible under disambiguation rules.
3. secondary_theme_1 / _2 are non-duplicative of primary and of each other.
4. sentiment uses ONLY Positive / Negative / Neutral / Unclear.
5. q1 exactly matches sentiment.
6. q2 exactly matches emotion.
7. tone is delivery, distinct from emotion.
8. Every "Yes" gate (q4, q12-q21) has explicit textual evidence.
9. All conditional fields are empty when the gate is "No" / not applicable.
10. No invented labels, no generic placeholder open-text.

────────────────────────────────────────────────────────────────────
OUTPUT — JSON ONLY, no markdown, no commentary
────────────────────────────────────────────────────────────────────
{{
  "primary_theme": "",
  "secondary_theme_1": "",
  "secondary_theme_2": "",
  "masculinity_identity": "",
  "normative_orientation": "",
  "target_of_claim": "",
  "sentiment": "",
  "emotion": "",
  "tone": "",
  "q1": "", "q2": "", "q3": [], "q4": "", "q5": "", "q6": "",
  "q7": [], "q7a": "",
  "q8": "", "q9": "", "q10": [], "q11": "",
  "q12": "", "q13": "",
  "q14": "", "q14a": "", "q15": "", "q15a": "", "q16": "", "q16a": "",
  "q17": "", "q17a": "", "q18": "", "q18a": "", "q19": "", "q19a": "",
  "q20": "", "q20a": "",
  "q21": "", "q21a": "", "q21b": "", "q21c": "", "q21d": "",
  "q21e": "", "q21f": "", "q21g": "", "q21h": ""
}}
"""
print(f'prompt length: {len(PROMPT):,} chars')

prompt length: 13,216 chars


## 4 — Run async (cached by SHA-1 of comment text)

In [5]:
def text_hash(s: str) -> str:
    return hashlib.sha1(s.encode('utf-8')).hexdigest()[:16]

def load_cache():
    if CACHE.exists():
        c = pd.read_parquet(CACHE)
        return {r.text_hash: json.loads(r.result_json) for r in c.itertuples()}
    return {}

def save_cache(cache):
    rows = [{'text_hash': k, 'result_json': json.dumps(v, ensure_ascii=False)} for k, v in cache.items()]
    pd.DataFrame(rows).to_parquet(CACHE, index=False)

async def code_one(client, sem, text):
    async with sem:
        try:
            r = await client.chat.completions.create(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': PROMPT},
                    {'role': 'user', 'content': text[:2000]},
                ],
                temperature=0,
                max_tokens=1200,
                response_format={'type': 'json_object'},
            )
            return json.loads(r.choices[0].message.content)
        except Exception as e:
            return {'error': str(e)[:200]}

async def run_all():
    cache = load_cache()
    print(f'cache: {len(cache)} entries')
    todo = [(text_hash(t), t) for t in sample['text_english'] if text_hash(t) not in cache]
    print(f'rows to code: {len(todo)} / {len(sample)}')
    if not todo:
        return cache
    client = AsyncOpenAI()
    sem = asyncio.Semaphore(CONCURRENCY)
    tasks = [code_one(client, sem, t) for _, t in todo]
    results = await atqdm.gather(*tasks)
    for (h, _), res in zip(todo, results):
        cache[h] = res
    save_cache(cache)
    print(f'cached: {len(cache)} entries')
    return cache

cache = await run_all()

cache: 200 entries
rows to code: 0 / 200


## 5 — Validation: enforce closed-vocab, repair invalid options

In [6]:
import json as _json

# Load the verbatim human-codebook headers (we will use these as xlsx column names)
with open(ROOT / 'temp' / 'human_audience_headers.json') as f:
    HUMAN_HDRS = _json.load(f)
# index by Q-number prefix for safe lookup
HDR_BY_KEY = {}
for h in HUMAN_HDRS:
    first = h.split('\n')[0].strip()
    # 'Q1.', 'Q2.', 'Q7A.', 'Q14a.' etc → key = first token before first '.' or first space, normalised
    if first.startswith('Q'):
        # take everything up to first '.' or first space
        if '.' in first:
            key = first.split('.')[0].strip().lower().replace(' ', '')
        else:
            key = first.split()[0].lower()
        HDR_BY_KEY[key] = h
    else:
        HDR_BY_KEY[first] = h

def hdr(q: str) -> str:
    """Return exact verbatim header from human codebook for a Q-key like 'q1' or 'q21h'."""
    h = HDR_BY_KEY.get(q.lower())
    assert h is not None, f"missing header for {q}"
    return h

# allowed value sets
Q1_OPTS = ['Positive', 'Negative', 'Neutral', 'Unclear']
Q3_OPTS = ['Feeling seen/understood', 'Feeling unseen/misunderstood',
           'Feeling attacked', 'Feeling objectified', 'None of these']
Q4_OPTS = ['Yes', 'No']
Q5_OPTS = ['Positive', 'Negative', 'Neutral', 'Unclear', 'Does not mention men/masculinity']
Q6_OPTS = ['Positive', 'Negative', 'Neutral', 'Unclear', 'Does not mention women/femininity']
Q7_OPTS = ['The speaker/creator of the content/influencer/the content',
           'Politics/social issues', 'Dating/relationships/marriage',
           'Money/status', 'Fitness', 'Media/video games',
           'Mental health/emotions', 'Gender roles/norms', 'Other']
Q8_OPTS = ['Supporting', 'Challenging', 'Neutral', 'Unclear']
Q10_OPTS = ['Entertainment/escapism', 'Information seeking',
            'Connection/social interaction',
            'Self expression/identity construction', 'Status seeking',
            'Documentation of events', 'None of these apply']
YN = ['Yes', 'No']
Q21H_OPTS = ['Female', 'Male', 'Non-binary', 'Other', 'Unclear']

def coerce_one(val, opts, default=''):
    if not isinstance(val, str): return default
    v = val.strip()
    if v in opts: return v
    for o in opts:
        if v.lower() == o.lower(): return o
    return default

def coerce_multi(val, opts):
    if not isinstance(val, list): return []
    seen = []
    for v in val:
        c = coerce_one(v, opts, default='')
        if c and c not in seen:
            seen.append(c)
    return seen

def sanitize_themes(primary, sec1, sec2):
    """Coerce + apply non-duplication + Unclear-clears-secondaries rule."""
    p = coerce_one(primary, THEMES, 'Unclear')
    s1 = coerce_one(sec1, THEMES, '')
    s2 = coerce_one(sec2, THEMES, '')
    if p == 'Unclear':
        return p, '', ''
    if s1 == p: s1 = ''
    if s2 == p or s2 == s1: s2 = ''
    return p, s1, s2

# build rows with EXACT human-codebook column headers
rows = []
coerce_log = {'q1_q2_locked': 0, 'theme_dup_dropped': 0, 'invalid_options_default': 0}
for _, r in sample.iterrows():
    h = text_hash(r['text_english'])
    raw = cache.get(h, {})

    # themes
    p, s1, s2 = sanitize_themes(
        raw.get('primary_theme'),
        raw.get('secondary_theme_1'),
        raw.get('secondary_theme_2'),
    )
    themes_combined = '; '.join([t for t in [p, s1, s2] if t])

    # front summary
    sentiment = coerce_one(raw.get('sentiment'), SENTIMENTS, 'Unclear')
    emotion   = coerce_one(raw.get('emotion'), EMOTIONS, 'None of these')
    tone      = coerce_one(raw.get('tone'), TONES, 'Detached')

    # ENFORCE q1 == sentiment, q2 == emotion (chatgpt's rule)
    raw_q1 = coerce_one(raw.get('q1'), Q1_OPTS, sentiment)
    raw_q2 = coerce_one(raw.get('q2'), EMOTIONS, emotion)
    q1 = sentiment    # locked
    q2 = emotion      # locked
    if raw_q1 != sentiment or raw_q2 != emotion:
        coerce_log['q1_q2_locked'] += 1

    # other Qs
    q3  = coerce_multi(raw.get('q3'), Q3_OPTS)
    q3_str = '; '.join(q3) if q3 else 'None of these'
    q4  = coerce_one(raw.get('q4'), Q4_OPTS, 'No')
    q5  = coerce_one(raw.get('q5'), Q5_OPTS, 'Does not mention men/masculinity')
    q6  = coerce_one(raw.get('q6'), Q6_OPTS, 'Does not mention women/femininity')
    q7  = coerce_multi(raw.get('q7'), Q7_OPTS)
    q7_str = '; '.join(q7) if q7 else 'Other'
    q7a = (raw.get('q7a') or '').strip()
    if 'Other' not in q7: q7a = ''  # gate

    q8  = coerce_one(raw.get('q8'), Q8_OPTS, 'Unclear')
    q9  = (raw.get('q9') or '').strip();   q9  = q9  if q8 == 'Supporting' else ''
    q10 = coerce_multi(raw.get('q10'), Q10_OPTS)
    q10_str = '; '.join(q10) if q10 else 'None of these apply'
    if q8 != 'Supporting': q10_str = 'None of these apply'
    q11 = (raw.get('q11') or '').strip(); q11 = q11 if q8 == 'Challenging' else ''

    q12 = coerce_one(raw.get('q12'), YN, 'No')
    q13 = coerce_one(raw.get('q13'), YN, 'No')

    q14  = coerce_one(raw.get('q14'),  YN, 'No')
    q14a = (raw.get('q14a') or '').strip(); q14a = q14a if q14 == 'Yes' else ''
    q15  = coerce_one(raw.get('q15'),  YN, 'No')
    q15a = (raw.get('q15a') or '').strip(); q15a = q15a if q15 == 'Yes' else ''
    q16  = coerce_one(raw.get('q16'),  YN, 'No')
    q16a = (raw.get('q16a') or '').strip(); q16a = q16a if q16 == 'Yes' else ''
    q17  = coerce_one(raw.get('q17'),  YN, 'No')
    q17a = (raw.get('q17a') or '').strip(); q17a = q17a if q17 == 'Yes' else ''
    q18  = coerce_one(raw.get('q18'),  YN, 'No')
    q18a = (raw.get('q18a') or '').strip(); q18a = q18a if q18 == 'Yes' else ''
    q19  = coerce_one(raw.get('q19'),  YN, 'No')
    q19a = (raw.get('q19a') or '').strip(); q19a = q19a if q19 == 'Yes' else ''
    q20  = coerce_one(raw.get('q20'),  YN, 'No')
    q20a = (raw.get('q20a') or '').strip(); q20a = q20a if q20 == 'Yes' else ''
    q21  = coerce_one(raw.get('q21'),  YN, 'No')

    q21a = coerce_one(raw.get('q21a'), YN, 'No')
    q21b = (raw.get('q21b') or '').strip(); q21b = q21b if q21a == 'Yes' else ''
    q21c = coerce_one(raw.get('q21c'), YN, 'No')
    q21d = (raw.get('q21d') or '').strip(); q21d = q21d if q21c == 'Yes' else ''
    q21e = coerce_one(raw.get('q21e'), YN, 'No')
    q21f = (raw.get('q21f') or '').strip(); q21f = q21f if q21e == 'Yes' else ''
    q21g = coerce_one(raw.get('q21g'), YN, 'No')
    q21h = coerce_one(raw.get('q21h'), Q21H_OPTS, '')
    if q21g != 'Yes': q21h = ''

    rows.append({
        'Comment ID':                r['comment_id'],
        'Commenter Post URL':        r['Commenter Post URL'],
        "Influencer's OG Post URL":  r["Influencer's OG Post URL"],
        'Comment Text':              r['text_english'],
        'Themes':                    themes_combined,
        'Sentiment':                 sentiment,
        'Emotion Detection':         emotion,
        'Tone':                      tone,
        hdr('q1'):    q1,
        hdr('q2'):    q2,
        hdr('q3'):    q3_str,
        hdr('q4'):    q4,
        hdr('q5'):    q5,
        hdr('q6'):    q6,
        hdr('q7'):    q7_str,
        hdr('q7a'):   q7a,
        hdr('q8'):    q8,
        hdr('q9'):    q9,
        hdr('q10'):   q10_str,
        hdr('q11'):   q11,
        hdr('q12'):   q12,
        hdr('q13'):   q13,
        hdr('q14'):   q14,
        hdr('q14a'):  q14a,
        hdr('q15'):   q15,
        hdr('q15a'):  q15a,
        hdr('q16'):   q16,
        hdr('q16a'):  q16a,
        hdr('q17'):   q17,
        hdr('q17a'):  q17a,
        hdr('q18'):   q18,
        hdr('q18a'):  q18a,
        hdr('q19'):   q19,
        hdr('q19a'):  q19a,
        hdr('q20'):   q20,
        hdr('q20a'):  q20a,
        hdr('q21'):   q21,
        hdr('q21a'):  q21a,
        hdr('q21b'):  q21b,
        hdr('q21c'):  q21c,
        hdr('q21d'):  q21d,
        hdr('q21e'):  q21e,
        hdr('q21f'):  q21f,
        hdr('q21g'):  q21g,
        hdr('q21h'):  q21h,
        'creator':     r['creator'],
        'orientation': r['orientation'],
    })
out = pd.DataFrame(rows)
print(f'coded rows: {len(out)}; q1/q2 locks applied to {coerce_log["q1_q2_locked"]} rows')
print(f'\nfinal column count: {len(out.columns)} (expected 45 + 2 helper = 47)')
print(f'\nSentiment dist: {out["Sentiment"].value_counts().to_dict()}')
print(f'Tone dist:      {out["Tone"].value_counts().to_dict()}')
print(f'Emotion dist:   {out["Emotion Detection"].value_counts().to_dict()}')
print(f'Themes (primary only) dist:')
print(out['Themes'].apply(lambda s: s.split(';')[0].strip() if s else 'Unclear').value_counts().to_string())

coded rows: 200; q1/q2 locks applied to 0 rows

final column count: 47 (expected 45 + 2 helper = 47)

Sentiment dist: {'Negative': 149, 'Positive': 37, 'Neutral': 14}
Tone dist:      {'Hostile': 108, 'Earnest': 64, 'Authoritative': 17, 'Detached': 7, 'Sarcastic': 3, 'Humorous': 1}
Emotion dist:   {'Anger': 133, 'Hope': 22, 'None of these': 14, 'Contempt': 10, 'Happiness': 9, 'Joy': 5, 'Fear': 3, 'Sadness': 3, 'Empathy': 1}
Themes (primary only) dist:
Themes
Gender Grievance                     63
Male Victimhood                      37
Gender-Based Violence and Consent    23
Sexual Morality                      21
Egalitarian Partnership              13
Marriage and Family                  10
Authority and Submission             10
Male Accountability                   9
Relationship Tactics                  6
Unclear                               5
Provider and Status                   2
Self-Discipline                       1


## 6 — Summary stats by creator + orientation

In [7]:
print('=== SENTIMENT × creator ===')
print(out.groupby(['creator','Sentiment']).size().unstack(fill_value=0).to_string())
print('\n=== TONE × creator ===')
print(out.groupby(['creator','Tone']).size().unstack(fill_value=0).to_string())
print('\n=== EMOTION × orientation ===')
print(out.groupby(['orientation','Emotion Detection']).size().unstack(fill_value=0).to_string())

# theme frequency (primary only — first in semicolon list)
print('\n=== top themes (primary) ===')
first_theme = out['Themes'].apply(lambda s: s.split(';')[0].strip() if s else 'Unclear')
print(first_theme.value_counts().to_string())

# stance × creator (Q8)
q8_col = hdr('q8')
print(f'\n=== stance ({q8_col[:60]}...) × creator ===')
print(out.groupby(['creator', q8_col]).size().unstack(fill_value=0).to_string())

=== SENTIMENT × creator ===
Sentiment         Negative  Neutral  Positive
creator                                      
Agba John Doe           46        0         4
Banky Wellington        14        8        28
Deyemi Okanlawon        44        4         2
Shola                   45        2         3

=== TONE × creator ===
Tone              Authoritative  Detached  Earnest  Hostile  Humorous  Sarcastic
creator                                                                         
Agba John Doe                10         0       13       26         0          1
Banky Wellington              5         4       35        5         1          0
Deyemi Okanlawon              0         1       11       38         0          0
Shola                         2         2        5       39         0          2

=== EMOTION × orientation ===
Emotion Detection  Anger  Contempt  Empathy  Fear  Happiness  Hope  Joy  None of these  Sadness
orientation                                                

## 7 — Export to xlsx

In [8]:
import openpyxl
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

# drop helper cols from the export
export_cols = [c for c in out.columns if c not in ('creator', 'orientation')]
out_export = out[export_cols].copy()

with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as xw:
    out_export.to_excel(xw, sheet_name='LLM Coding', index=False)
    summary = pd.DataFrame({
        'metric': ['Total rows', 'Creators', 'Per creator', 'Model', 'Seed',
                   'Themes vocabulary', 'Sentiment values', 'Emotion values', 'Tone values'],
        'value':  [len(out_export), out['creator'].nunique(), N_PER_CREATOR, MODEL, SEED,
                   ', '.join(THEMES), ', '.join(SENTIMENTS),
                   ', '.join(EMOTIONS), ', '.join(TONES)],
    })
    summary.to_excel(xw, sheet_name='Methodology', index=False)

# pretty-format the xlsx
wb = openpyxl.load_workbook(OUT_XLSX)
ws = wb['LLM Coding']
header_fill = PatternFill('solid', fgColor='305496')
header_font = Font(bold=True, color='FFFFFF', size=10)
for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(wrap_text=True, vertical='center', horizontal='left')
ws.row_dimensions[1].height = 60
ws.freeze_panes = 'E2'
for col_idx, col_name in enumerate(out_export.columns, 1):
    letter = get_column_letter(col_idx)
    if col_name == 'Comment Text':
        ws.column_dimensions[letter].width = 60
    elif col_name in ('Commenter Post URL', "Influencer's OG Post URL"):
        ws.column_dimensions[letter].width = 35
    elif col_name == 'Themes':
        ws.column_dimensions[letter].width = 35
    elif col_name.startswith('Q') and col_name.endswith('a'):
        ws.column_dimensions[letter].width = 35
    else:
        ws.column_dimensions[letter].width = 18
for row in ws.iter_rows(min_row=2):
    for c in row:
        c.alignment = Alignment(wrap_text=True, vertical='top')
        c.font = Font(size=10)
wb.save(OUT_XLSX)
print(f'wrote {OUT_XLSX}  ({OUT_XLSX.stat().st_size:,} bytes)')

wrote /Users/sushildalavi/Desktop/NLC/Gates-Manfluencer-Project/Codebooks/LLM Codebook/LLM Coding - Audience Analysis.xlsx  (79,145 bytes)


## Notes

- Cache is keyed by SHA-1 of `text_english` and lives at `temp/llm_audience_coding_cache.parquet`. Re-running the notebook is free.
- To force a re-code, delete the cache file or change the prompt.
- Sample is 200 comments — 50 per creator, balanced 50/50 progressive vs regressive. To change, edit `N_PER_CREATOR` in setup. Maximum is 66 per creator (Shola has the smallest pool at 66 rows).
- All answers are validated against the closed vocabularies before export. Any LLM responses outside the allowed options are coerced to safe defaults (`Unclear`, `No`, `None of these`, etc.) — review the Sentiment / Q1 / Q4–Q21 columns for any defaults that should be flagged for human review.
- Sentiment uses the full 4-class human-codebook vocabulary: Positive / Negative / Neutral / Unclear.